In [ ]:
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df_flow = pd.read_csv('FlowFISH Data(T cell subsets RTL).csv', index_col=0)
df_flow['Donor'] = df_flow.index
pop_dfs = {pop: df_flow[df_flow.Population == pop]
           for pop in df_flow.Population.unique()}

In [ ]:
df_flow.keys()

In [ ]:
custom_palette = {
    'SPL': '#e41a1c',       # Explicit Red
    'LLN': '#377eb8',      # Explicit Blue
    'MLN': '#4daf4a',          # Explicit Green
    'LNG': '#984ea3'  # Explicit Purple
}

In [ ]:
from plotting_methods import RTL_boxplot

for key in pop_dfs.keys():
    df = pop_dfs[key]
    if df.Tissue.nunique() < 2:
        print(f"Skipping {key} due to insufficient tissue types.")
        continue

    RTL_boxplot(df, key)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from statannotations.Annotator import Annotator
from plotting_methods import population_boxplot

population_boxplot(df_flow)

In [ ]:
for key in pop_dfs.keys():
    df = pop_dfs[key]

In [ ]:
df = pop_dfs['Tregs'].sort_values('Age')
df

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

for key in pop_dfs.keys():
    df = pop_dfs[key].sort_values('Age')
    if df.Tissue.nunique() < 2:
        print(f"Skipping {key} due to insufficient tissue types.")
        continue

    custom_palette = {
        'SPL': '#e41a1c',       # Explicit Red
        'LLN': '#377eb8',      # Explicit Blue
        'MLN': '#4daf4a',          # Explicit Green
        'LNG': '#984ea3'  # Explicit Purple
    }

    fig, ax = plt.subplots(figsize=(7, 5))

    sns.scatterplot(
        x='Age', y='RTL', data=df,
        palette=custom_palette, hue='Tissue', s=60, edgecolor='black', linewidth=0.5
    )
    plt.plot(df['Age'], df['RTL'], marker='o',
             zorder=0, color='gray', alpha=0.5)

    sns.despine()
    plt.tight_layout()
    plt.grid()
    plt.title("RTL vs Age for {}".format(key), fontsize=14, pad=15)
    plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import TargetEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

model_df = df_flow[['RTL', 'Age', 'Tissue', 'Population']].dropna()
X = model_df[['Age', 'Tissue', 'Population']]
y = model_df['RTL']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

preprocess = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', ['Age']),
        ('cat', TargetEncoder(cv=5, random_state=42), ['Tissue', 'Population'])
    ]
)

reg_model = Pipeline([
    ('preprocess', preprocess),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1))
])

cv_strategy = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(reg_model, X, y, cv=cv_strategy, scoring='r2')

print(f"All CV R^2 scores: {cv_scores}")
print(f"Mean CV R^2: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}\n")

reg_model.fit(X_train, y_train)
y_pred_train = reg_model.predict(X_train)
y_pred_test = reg_model.predict(X_test)

print(f"R^2 (test set): {r2_score(y_test, y_pred_test):.4f}\n")

residuals_train = y_train - y_pred_train
residuals_test = y_test - y_pred_test

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_pred_train, y=residuals_train,
                color='blue', alpha=0.6, label='Train Data')
sns.scatterplot(x=y_pred_test, y=residuals_test,
                color='crimson', alpha=0.7, label='Test Data')
plt.axhline(y=0, color='black', linestyle='--', linewidth=1.5)

plt.title('Residual Plot: Predicted RTL vs. Residuals (Random Forest Model)')
plt.xlabel('Predicted Relative Telomere Length (RTL)')
plt.ylabel('Residuals (Actual - Predicted)')
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

importances = reg_model.named_steps['regressor'].feature_importances_
feature_names = ['Age', 'Tissue_encoded', 'Population_encoded']

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False).reset_index(drop=True)

display(importance_df)